In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
df = pd.read_csv('bureau_observations.csv')

In [ ]:
df.head()

In [ ]:
df2 = pd.read_csv('customer_profiles.csv')

In [ ]:
df2.head()

In [ ]:
df3 = pd.read_csv('ecommerce_transactions.csv')

In [ ]:
df3.head()

In [ ]:
df4 = pd.read_csv('previous_applications.csv')
df4.head()

In [ ]:
df5 = pd.read_csv('repayment_history.csv')

In [ ]:
df5.head()

In [ ]:
df6 = pd.read_csv('account_behavior.csv')
df6.head()

In [ ]:
df.columns.tolist()

In [ ]:
df2.columns.tolist()

In [ ]:
df3.columns.tolist()

In [ ]:
df4.columns.tolist()

In [ ]:
df5.columns.tolist()

In [ ]:
df6.columns.tolist()

In [ ]:

required = {
    "df":  ["application_id", "customer_id", "score", "utilization",
            "inquiries_last_6m", "delinquent_accounts_last_12m"],
    "df2": ["customer_id", "region", "employment_type", "declared_monthly_income"],
    "df3": ["customer_id", "transaction_datetime", "order_total"],
    "df4": ["application_id", "customer_id", "application_datetime",
            "decision", "loan_id", "requested_amount", "term_months"],
    "df5": ["customer_id", "due_date", "days_past_due"],
    "df6": ["customer_id", "month", "inflows", "outflows"],
}
dfs = {"df": df, "df2": df2, "df3": df3, "df4": df4, "df5": df5, "df6": df6}
for k, cols in required.items():
    missing = [c for c in cols if c not in dfs[k].columns]
    print(f"{k}: missing {missing}")

In [ ]:
# ---------- 1. Parse tanggal ----------
df["report_datetime"]     = pd.to_datetime(df["report_datetime"], errors="coerce")
df["available_datetime"]  = pd.to_datetime(df["available_datetime"], errors="coerce")

df3["transaction_datetime"] = pd.to_datetime(df3["transaction_datetime"], errors="coerce")

df4["application_datetime"] = pd.to_datetime(df4["application_datetime"], errors="coerce")
df4["decision_datetime"]    = pd.to_datetime(df4["decision_datetime"], errors="coerce")

df5["due_date"]            = pd.to_datetime(df5["due_date"], errors="coerce")
if "last_payment_datetime" in df5.columns:
    df5["last_payment_datetime"] = pd.to_datetime(df5["last_payment_datetime"], errors="coerce")

# month bisa "2024-01" atau datetime
df6["month"] = pd.to_datetime(df6["month"], errors="coerce")

In [ ]:
# ---------- 2. Base table: applications + customers ----------
base = df4.merge(df2, on="customer_id", how="left", validate="many_to_one")

base["origination_date"] = base["decision_datetime"].fillna(base["application_datetime"])
base["app_month"] = base["application_datetime"].dt.to_period("M").dt.to_timestamp()

# populasi target = approved + punya loan_id
base_approved = base[base["decision"].eq("approved") & base["loan_id"].notna()].copy()
print("Total apps:", len(base), "| Approved + loan:", len(base_approved))

In [ ]:
# ---------- 3. Target dari loan_installments ----------
# Cek dulu apakah df5 punya loan_id. Kalau tidak, skip blok ini.
if "loan_id" in df5.columns:
    # Ambil origination per loan
    loan_orig = (df5.groupby("loan_id")["due_date"].min()
                 .rename("first_due_date").reset_index())
    inst = df5.merge(loan_orig, on="loan_id", how="left")
    inst["months_since_orig"] = (
        (inst["due_date"].dt.year - inst["first_due_date"].dt.year) * 12
        + (inst["due_date"].dt.month - inst["first_due_date"].dt.month)
    )
    inst_12m = inst[inst["months_since_orig"].between(0, 12)]

    target = (inst_12m.groupby("loan_id")
              .agg(target=("days_past_due", lambda s: int((s >= 90).any())),
                   months_observed=("months_since_orig", "max"))
              .reset_index())

    # hanya loan mature >= 12 bulan
    target_mature = target[target["months_observed"] >= 12].copy()
    target_mature["target"] = target_mature["target"].astype(int)
    print("Loan mature:", len(target_mature), "| bad rate:", target_mature["target"].mean().round(4))
else:
    print("df5 tidak punya loan_id, perlu strategi lain")

In [ ]:
# ---------- 4. Bureau (agregasi per application_id) ----------
bureau_feat = (df.sort_values("available_datetime")
                 .groupby("application_id")
                 .agg(
                    bureau_score_max      = ("score", "max"),
                    bureau_score_min      = ("score", "min"),
                    bureau_score_mean     = ("score", "mean"),
                    bureau_active_accounts= ("active_accounts", "sum"),
                    bureau_outstanding    = ("outstanding_balance", "sum"),
                    bureau_rev_limit      = ("revolving_credit_limit", "sum"),
                    bureau_rev_outstanding= ("revolving_outstanding_balance", "sum"),
                    bureau_utilization    = ("utilization", "mean"),
                    bureau_utilization_max= ("utilization", "max"),
                    bureau_inquiries_6m   = ("inquiries_last_6m", "sum"),
                    bureau_delinq_12m     = ("delinquent_accounts_last_12m", "sum"),
                    bureau_provider_count = ("bureau_provider", "nunique"),
                    bureau_obs_count      = ("bureau_observation_id", "count"),
                 )
                 .reset_index())

bureau_feat["bureau_rev_util"] = (
    bureau_feat["bureau_rev_outstanding"]
    / bureau_feat["bureau_rev_limit"].replace(0, np.nan)
)

In [ ]:
# ---------- 5. Ecommerce (point-in-time, agregasi per application) ----------
ecom = df3.merge(
    base_approved[["application_id", "customer_id", "application_datetime"]],
    on="customer_id", how="inner"
)
ecom = ecom[ecom["transaction_datetime"] < ecom["application_datetime"]].copy()
ecom["days_before"] = (ecom["application_datetime"] - ecom["transaction_datetime"]).dt.days
ecom["order_total"] = pd.to_numeric(ecom["order_total"], errors="coerce")
ecom["item_total"]  = pd.to_numeric(ecom["item_total"], errors="coerce")
ecom["quantity"]    = pd.to_numeric(ecom["quantity"], errors="coerce")

def ecom_window(d, days, suf):
    w = d[d["days_before"] <= days]
    return w.groupby("application_id").agg(**{
        f"ecom_orders_{suf}":   ("order_id", "nunique"),
        f"ecom_spend_{suf}":    ("order_total", "sum"),
        f"ecom_avg_{suf}":      ("order_total", "mean"),
        f"ecom_max_{suf}":      ("order_total", "max"),
        f"ecom_qty_{suf}":      ("quantity", "sum"),
    })

ecom_feat = (ecom_window(ecom, 30, "30d")
             .join(ecom_window(ecom, 90, "90d"), how="outer")
             .join(ecom_window(ecom, 180, "180d"), how="outer")
             .join(ecom_window(ecom, 365, "365d"), how="outer")
             .reset_index())

# Recency
last_ord = (ecom.groupby("application_id")["transaction_datetime"].max()
            .rename("ecom_last_dt").reset_index())
last_ord = last_ord.merge(base_approved[["application_id","application_datetime"]],
                          on="application_id", how="left")
last_ord["ecom_recency_days"] = (last_ord["application_datetime"] - last_ord["ecom_last_dt"]).dt.days
ecom_feat = ecom_feat.merge(last_ord[["application_id","ecom_recency_days"]],
                            on="application_id", how="left")

# Kategori & payment method favorit
top_cat = (ecom.groupby(["application_id","product_category"]).size()
           .reset_index(name="n").sort_values("n", ascending=False)
           .drop_duplicates("application_id")
           .rename(columns={"product_category":"ecom_top_category"})[["application_id","ecom_top_category"]])
ecom_feat = ecom_feat.merge(top_cat, on="application_id", how="left")

top_pay = (ecom.groupby(["application_id","payment_method"]).size()
           .reset_index(name="n").sort_values("n", ascending=False)
           .drop_duplicates("application_id")
           .rename(columns={"payment_method":"ecom_top_payment"})[["application_id","ecom_top_payment"]])
ecom_feat = ecom_feat.merge(top_pay, on="application_id", how="left")

# unique category
uc = (ecom.groupby("application_id")["product_category"].nunique()
      .rename("ecom_unique_categories").reset_index())
ecom_feat = ecom_feat.merge(uc, on="application_id", how="left")

In [ ]:
# ---------- 6. Account behavior (point-in-time) ----------
bank = df6.merge(
    base_approved[["application_id","customer_id","app_month"]],
    on="customer_id", how="inner"
)
bank = bank[bank["month"] < bank["app_month"]].copy()
bank["months_before"] = (
    (bank["app_month"].dt.year - bank["month"].dt.year) * 12
    + (bank["app_month"].dt.month - bank["month"].dt.month)
)

def bank_window(d, months, suf):
    w = d[d["months_before"] <= months]
    g = w.groupby("application_id").agg(**{
        f"bank_inflows_{suf}":   ("inflows", "mean"),
        f"bank_outflows_{suf}":  ("outflows", "mean"),
        f"bank_income_{suf}":    ("income_like_credits", "mean"),
        f"bank_avg_bal_{suf}":   ("average_balance", "mean"),
        f"bank_min_bal_{suf}":   ("minimum_balance", "min"),
        f"bank_low_days_{suf}":  ("low_balance_days", "sum"),
        f"bank_months_{suf}":    ("month", "nunique"),
    })
    g[f"bank_netflow_{suf}"] = g[f"bank_inflows_{suf}"] - g[f"bank_outflows_{suf}"]
    g[f"bank_oir_{suf}"] = g[f"bank_outflows_{suf}"] / g[f"bank_inflows_{suf}"].replace(0, np.nan)
    return g

bank_feat = (bank_window(bank, 3, "3m")
             .join(bank_window(bank, 6, "6m"), how="outer")
             .join(bank_window(bank, 12, "12m"), how="outer")
             .reset_index())

# slope saldo
def slope(g):
    if len(g) < 2: return np.nan
    x = g["months_before"].to_numpy(dtype=float)
    y = g["closing_balance"].to_numpy(dtype=float)
    return float(np.polyfit(x, y, 1)[0])

slope_df = (bank.sort_values("months_before")
            .groupby("application_id")
            .apply(slope, include_groups=False)
            .rename("bank_balance_slope").reset_index())
bank_feat = bank_feat.merge(slope_df, on="application_id", how="left")

In [ ]:
# ---------- 7. Feature aplikasi ----------
base_approved["loan_to_income"] = (
    base_approved["requested_amount"] / base_approved["declared_monthly_income"].replace(0, np.nan)
)
base_approved["installment_to_income"] = (
    base_approved["scheduled_monthly_installment"] / base_approved["declared_monthly_income"].replace(0, np.nan)
)
base_approved["approved_ratio"] = (
    base_approved["approved_amount"] / base_approved["requested_amount"].replace(0, np.nan)
)
base_approved["income_per_dep"] = (
    base_approved["declared_monthly_income"] / (base_approved["dependents"].fillna(0) + 1)
)

In [ ]:
# ---------- 8. Master ----------
master = (base_approved
          .merge(bureau_feat, on="application_id", how="left")
          .merge(ecom_feat,   on="application_id", how="left")
          .merge(bank_feat,   on="application_id", how="left")
          .merge(target_mature[["loan_id","target"]], on="loan_id", how="inner"))

# indikator ketersediaan
master["has_bureau"] = master["bureau_obs_count"].notna().astype(int)
master["has_ecom"]   = master["ecom_orders_180d"].notna().astype(int)
master["has_bank"]   = master["bank_months_6m"].notna().astype(int)

# isi 0 untuk count/sum
zero_cols = [c for c in master.columns
             if any(k in c for k in ["ecom_orders_","ecom_qty_","bank_low_days_",
                                      "bureau_inquiries_","bureau_delinq_",
                                      "bureau_active_accounts"])]
master[zero_cols] = master[zero_cols].fillna(0)

print("Master shape:", master.shape)
print("Bad rate:", master["target"].mean().round(4))
master.to_pickle("master_train.pkl")
master = pd.read_pickle("master_train.pkl")

In [ ]:

master = pd.read_pickle("master_train.pkl")
print("Shape:", master.shape)
print("Bad rate:", master["target"].mean().round(4))
print("Kolom tanggal:", master.select_dtypes(include=["datetime64[ns]"]).columns.tolist())

In [ ]:
master.head()

In [ ]:
leak_keywords = ["days_past_due", "amount_paid", "last_payment", "installment_id",
                 "max_days_past_due", "loan_id", "order_id", "installment"]
leak_cols = [c for c in master.columns if any(k in c.lower() for k in leak_keywords)]
print("Kolom berpotensi leakage:", leak_cols)

In [ ]:
id_cols = ["application_id", "customer_id", "bureau_observation_id", "order_id", "order_item_id"]
drop_cols = list(set(leak_cols + id_cols + ["target"]))
feature_cols = [c for c in master.columns if c not in drop_cols]
print("Jumlah fitur:", len(feature_cols))

In [ ]:
nunique = master[feature_cols].nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()
print("Kolom konstan:", constant_cols)

# near-constant: satu nilai >99%
near_const = []
for c in feature_cols:
    top = master[c].value_counts(dropna=False, normalize=True).iloc[0]
    if top > 0.99:
        near_const.append((c, round(top, 4)))
print("Near-constant:", near_const[:20])

In [ ]:
missing = master[feature_cols].isna().mean().sort_values(ascending=False)
print("Missing > 50%:")
print(missing[missing > 0.5])

In [ ]:
# 1. Flag grup
master["has_bank"]  = master["bank_months_6m"].notna().astype(int)
master["has_ecom"]  = master["ecom_orders_180d"].notna().astype(int)
master["has_bureau_util"] = master["bureau_utilization"].notna().astype(int)

# 2. Isi 0 untuk count/sum
zero_fill = [c for c in master.columns if any(k in c for k in [
    "ecom_orders_", "ecom_qty_", "bank_low_days_", "bank_months_",
    "bureau_inquiries_", "bureau_delinq_", "bureau_active_accounts",
    "bureau_obs_count",
])]
master[zero_fill] = master[zero_fill].fillna(0)

# 3. Biarkan NaN untuk rasio, rata-rata, utilization, score

# 4. Drop kolom redundan
drop_redundant = [
    "bank_months_3m", "bank_months_6m",   # cukup 12m
    "bank_inflows_3m", "bank_outflows_3m",  # cukup 6m/12m
    "ecom_avg_30d", "ecom_spend_30d",       # 30d terlalu noisy
    "ecom_max_30d",
]
master = master.drop(columns=[c for c in drop_redundant if c in master.columns])

# 5. Simpan
master.to_pickle("master_train_clean.pkl")

In [ ]:
cols = master.columns.tolist()
for kw in ["ecom_", "bank_", "bureau_"]:
    print(f"=== {kw} ===")
    print([c for c in cols if c.startswith(kw)])

In [ ]:
id_cols = ["application_id", "customer_id", "bureau_observation_id",
           "order_id", "order_item_id", "loan_id"]
drop_cols = set(id_cols + ["target", "application_datetime", "decision_datetime",
                            "origination_date", "app_month"])

feature_cols = [c for c in master.columns if c not in drop_cols]
print("Jumlah fitur:", len(feature_cols))

In [ ]:
missing_after = master[feature_cols].isna().mean().sort_values(ascending=False)
print(missing_after[missing_after > 0.5])
print("Shape:", master.shape)
print("Bad rate:", master["target"].mean().round(4))

In [ ]:
master["has_bank"]  = master["bank_months_12m"].notna().astype(int) if "bank_months_12m" in master.columns else master["bank_avg_bal_6m"].notna().astype(int)
master["has_ecom"]  = master["ecom_orders_180d"].notna().astype(int)
master["has_ecom_90d"] = master["ecom_orders_90d"].notna().astype(int)
master["has_bureau_util"] = master["bureau_utilization"].notna().astype(int)

print(master.groupby("has_bank")["target"].agg(["mean", "count"]))
print(master.groupby("has_ecom")["target"].agg(["mean", "count"]))
print(master.groupby("has_ecom_90d")["target"].agg(["mean", "count"]))
print(master.groupby("has_bureau_util")["target"].agg(["mean", "count"]))

In [ ]:
print(master["bank_months_12m"].isna().sum())
print(master["bank_months_12m"].head(20))
print(master["ecom_orders_180d"].isna().sum())

In [ ]:
# Kolom count sudah di-fill 0, jadi pakai > 0
master["has_bank"]     = (master["bank_months_12m"] > 0).astype(int)
master["has_ecom"]     = (master["ecom_orders_180d"] > 0).astype(int)
master["has_ecom_90d"] = (master["ecom_orders_90d"] > 0).astype(int)

# bureau_utilization tidak di-fill 0, jadi notna() valid
master["has_bureau_util"] = master["bureau_utilization"].notna().astype(int)

In [ ]:
for col in ["has_bank", "has_ecom", "has_ecom_90d", "has_bureau_util"]:
    print(f"\n=== {col} ===")
    print(master.groupby(col)["target"].agg(["mean", "count"]))

In [ ]:
print(master.groupby(["has_bank", "has_bureau_util"])["target"].agg(["mean", "count"]))
print(master.groupby(["has_ecom", "has_bank"])["target"].agg(["mean", "count"]))

In [ ]:
master.to_pickle("master_train_clean.pkl")
print("Shape:", master.shape)
print("Bad rate:", master["target"].mean().round(4))

In [ ]:
!pip install lightgbm catboost shap matplotlib scipy

In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ============ LOAD MASTER ============
master = pd.read_pickle("master_train_clean.pkl")
print("Shape awal:", master.shape)
print("Bad rate:", master["target"].mean().round(4))

# ============ TIER 1: Interaksi bureau_score ============
master["score_x_loan_income"]   = master["bureau_score_mean"] * master["loan_to_income"]
master["score_x_installment"]   = master["bureau_score_mean"] * master["installment_to_income"]
master["score_x_util"]          = master["bureau_score_mean"] * master["bureau_utilization"]
master["score_x_delinq"]        = master["bureau_score_mean"] * master["bureau_delinq_12m"]
master["score_x_inquiry"]       = master["bureau_score_mean"] * master["bureau_inquiries_6m"]
master["score_div_loan_income"] = master["bureau_score_mean"] / (master["loan_to_income"] + 1)

# ============ TIER 2: Rasio turunan ============
master["installment_per_term"]  = master["approved_amount"] / master["term_months"].replace(0, np.nan)
master["inquiries_per_account"] = master["bureau_inquiries_6m"] / (master["bureau_active_accounts"] + 1)
master["balance_per_active"]    = master["bureau_outstanding"] / (master["bureau_active_accounts"] + 1)
master["income_vs_bureau"]      = master["declared_monthly_income"] / (master["bureau_outstanding"] + 1)
master["ecom_per_income"]       = master["ecom_spend_365d"] / (master["declared_monthly_income"] + 1)
master["delinq_per_account"]    = master["bureau_delinq_12m"] / (master["bureau_active_accounts"] + 1)
master["approved_per_income"]   = master["approved_amount"] / (master["declared_monthly_income"] + 1)

# ============ TIER 3: Log transform ============
for c in ["loan_to_income", "installment_to_income", "bureau_outstanding",
          "requested_amount", "approved_amount"]:
    if c in master.columns:
        master[f"log_{c}"] = np.log1p(master[c].clip(lower=0))

# ============ TIER 4: Demografi x kredit ============
master["age_x_tenure"]   = master["age"] * master["employment_tenure_months"]
master["age_x_score"]    = master["age"] * master["bureau_score_mean"]
master["tenure_x_score"] = master["employment_tenure_months"] * master["bureau_score_mean"]
master["income_per_age"] = master["declared_monthly_income"] / (master["age"] + 1)

# ============ TIER 5: E-commerce trend (defensif) ============
if "ecom_recency_days" in master.columns:
    master["ecom_recency_norm"] = master["ecom_recency_days"] / 365

print("Shape setelah FE:", master.shape)   # harus 114

In [ ]:
id_cols = ["application_id", "customer_id", "bureau_observation_id",
           "order_id", "order_item_id", "loan_id"]
meta_cols = ["target", "application_datetime", "decision_datetime",
             "origination_date", "app_month"]
leak_cols = ["decision"]                    # ← ini yang belum ke-apply

drop_cols = set(id_cols + meta_cols + leak_cols)

feature_cols = [c for c in master.columns if c not in drop_cols]
cat_cols = master[feature_cols].select_dtypes(include=["object","category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Fitur: {len(feature_cols)} | Cat: {len(cat_cols)} | Num: {len(num_cols)}")
print("Kategorikal:", cat_cols)
for c in cat_cols:
    print(f"  {c}: {master[c].nunique()} unique")

# Cek kolom konstan
nunique = master[feature_cols].nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()
print("Kolom konstan:", constant_cols)

In [ ]:
new_feats = [c for c in master.columns if any(k in c for k in
    ["score_x_", "score_div_", "_per_", "income_vs_", "ecom_per_",
     "log_", "age_x_", "tenure_x_", "income_per_age", "ecom_recency_norm"])]
new_in_features = [c for c in new_feats if c in feature_cols]
print(f"Fitur baru: {len(new_feats)}")
print(f"Fitur baru di feature_cols: {len(new_in_features)}")

In [ ]:
master = master.sort_values("application_datetime").reset_index(drop=True)
q1 = master["application_datetime"].quantile(0.70)
q2 = master["application_datetime"].quantile(0.85)

train = master[master["application_datetime"] <= q1].copy()
valid = master[(master["application_datetime"] > q1) & (master["application_datetime"] <= q2)].copy()
test  = master[master["application_datetime"] > q2].copy()

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:>5}: n={len(d):>6} | bad={d['target'].mean():.4f}")

X_train = train[feature_cols].copy(); y_train = train["target"].values
X_valid = valid[feature_cols].copy(); y_valid = valid["target"].values
X_test  = test[feature_cols].copy();  y_test  = test["target"].values

In [ ]:
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             brier_score_loss, confusion_matrix)

RESULTS_FE = []

def evaluate(y, p, name=""):
    auc   = roc_auc_score(y, p)
    gini  = 2*auc - 1
    ap    = average_precision_score(y, p)
    brier = brier_score_loss(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    ks    = float(np.max(tpr - fpr))
    print(f"[{name:>18}] AUC={auc:.4f} | Gini={gini:.4f} | KS={ks:.4f} | "
          f"PR-AUC={ap:.4f} | Brier={brier:.4f}")
    RESULTS_FE.append(dict(name=name, auc=auc, gini=gini, ks=ks, ap=ap, brier=brier))
    return auc

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_pipe_lr = Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("sc", StandardScaler())])
cat_pipe_lr = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20))])
pre_lr = ColumnTransformer([("num", num_pipe_lr, num_cols),
                            ("cat", cat_pipe_lr, cat_cols)])

num_pipe_tree = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_pipe_tree = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="__NA__")),
                          ("ord", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                 unknown_value=-1))])
pre_tree = ColumnTransformer([("num", num_pipe_tree, num_cols),
                              ("cat", cat_pipe_tree, cat_cols)])

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_fe = Pipeline([("pre", pre_lr),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5))])
lr_fe.fit(X_train, y_train)
p_lr_va = lr_fe.predict_proba(X_valid)[:, 1]
p_lr_te = lr_fe.predict_proba(X_test)[:, 1]
evaluate(y_valid, p_lr_va, "LR+FE valid")
evaluate(y_test,  p_lr_te, "LR+FE test")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_fe = Pipeline([("pre", pre_tree),
                  ("clf", RandomForestClassifier(n_estimators=500, max_depth=12,
                                                 min_samples_leaf=20,
                                                 class_weight="balanced_subsample",
                                                 n_jobs=-1, random_state=42))])
rf_fe.fit(X_train, y_train)
p_rf_va = rf_fe.predict_proba(X_valid)[:, 1]
p_rf_te = rf_fe.predict_proba(X_test)[:, 1]
evaluate(y_valid, p_rf_va, "RF+FE valid")
evaluate(y_test,  p_rf_te, "RF+FE test")

In [ ]:
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder

enc_x = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_x = X_train.copy(); X_valid_x = X_valid.copy(); X_test_x = X_test.copy()
X_train_x[cat_cols] = enc_x.fit_transform(X_train_x[cat_cols].astype(str))
X_valid_x[cat_cols] = enc_x.transform(X_valid_x[cat_cols].astype(str))
X_test_x[cat_cols]  = enc_x.transform(X_test_x[cat_cols].astype(str))

spw = (y_train == 0).sum() / (y_train == 1).sum()

xgb_fe = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.03, max_depth=4,
    min_child_weight=20, subsample=0.9, colsample_bytree=0.7,
    reg_alpha=1.0, reg_lambda=5.0, gamma=0.5,
    objective="binary:logistic", eval_metric="auc",
    scale_pos_weight=spw, tree_method="hist",
    enable_categorical=False, n_jobs=-1, random_state=42,
    early_stopping_rounds=50,
)
xgb_fe.fit(X_train_x, y_train, eval_set=[(X_valid_x, y_valid)], verbose=False)

p_xgb_va = xgb_fe.predict_proba(X_valid_x)[:, 1]
p_xgb_te = xgb_fe.predict_proba(X_test_x)[:, 1]
evaluate(y_valid, p_xgb_va, "XGB+FE valid")
evaluate(y_test,  p_xgb_te, "XGB+FE test")

In [ ]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression

def neg_ks(weights, preds, y):
    w = np.abs(weights) / np.abs(weights).sum()
    p = sum(wi * pi for wi, pi in zip(w, preds))
    return -(2 * roc_auc_score(y, p) - 1)

preds_va = [p_lr_va, p_rf_va, p_xgb_va]
preds_te = [p_lr_te, p_rf_te, p_xgb_te]

res = minimize(neg_ks, x0=[0.3, 0.4, 0.3], args=(preds_va, y_valid),
               method="Nelder-Mead", options={"xatol": 1e-4, "maxiter": 500})
w_opt = np.abs(res.x) / np.abs(res.x).sum()
print("Bobot optimal:", dict(zip(["LR","RF","XGB"], w_opt.round(3))))

p_blend_va = sum(wi * pi for wi, pi in zip(w_opt, preds_va))
p_blend_te = sum(wi * pi for wi, pi in zip(w_opt, preds_te))
evaluate(y_valid, p_blend_va, "blend+FE valid")
evaluate(y_test,  p_blend_te, "blend+FE test")

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_blend_va, y_valid)
p_blend_va_cal = iso.predict(p_blend_va)
p_blend_te_cal = iso.predict(p_blend_te)
evaluate(y_valid, p_blend_va_cal, "blend+FE-cal valid")
evaluate(y_test,  p_blend_te_cal, "blend+FE-cal test")

In [ ]:
from sklearn.isotonic import IsotonicRegression

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_blend_va, y_valid)

p_blend_va_cal = iso.predict(p_blend_va)
p_blend_te_cal = iso.predict(p_blend_te)

evaluate(y_valid, p_blend_va_cal, "blend-cal valid")
evaluate(y_test,  p_blend_te_cal, "blend-cal test")

In [ ]:
from sklearn.calibration import calibration_curve

frac, mean_p = calibration_curve(y_valid, p_blend_va_cal,
                                  n_bins=15, strategy="quantile")
print("Predicted → Observed")
for mp, f in zip(mean_p, frac):
    print(f"  {mp:.4f} → {f:.4f}")

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
models = [("LR", p_lr_va), ("HGB", p_hgb_va), ("RF", p_rf_va), ("Blend", p_blend_va)]

for name, p in models:
    RocCurveDisplay.from_predictions(y_valid, p, ax=axes[0], name=name)
axes[0].set_title("ROC — valid")

for name, p in models:
    PrecisionRecallDisplay.from_predictions(y_valid, p, ax=axes[1], name=name)
axes[1].set_title("PR — valid")

frac, mean_p = calibration_curve(y_valid, p_blend_va, n_bins=15, strategy="quantile")
axes[2].plot(mean_p, frac, marker="o", label="Blend")
axes[2].plot([0,1],[0,1],"--", color="gray")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("Observed")
axes[2].set_title("Calibration — valid"); axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
t_opt = best_threshold(y_valid, p_blend_va_cal, cost_fn=10, cost_fp=1)

yp = (p_blend_va_cal >= t_opt).astype(int)
tn, fp, fn, tp = confusion_matrix(y_valid, yp, labels=[0,1]).ravel()
print(f"\nValid @ {t_opt:.3f}: TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"Precision={tp/(tp+fp):.4f} | Recall={tp/(tp+fn):.4f}")
print(f"Approval rate={(tn+fn)/len(y_valid):.4f} | "
      f"Bad rate approved={fp/(tn+fp):.4f}")

In [ ]:
for cost_fn, cost_fp in [(5,1), (10,1), (20,1), (50,1)]:
    print(f"\n--- cost_fn={cost_fn}, cost_fp={cost_fp} ---")
    t = best_threshold(y_valid, p_blend_va_cal, cost_fn=cost_fn, cost_fp=cost_fp)
    yp = (p_blend_va_cal >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, yp, labels=[0,1]).ravel()
    print(f"  precision={tp/(tp+fp):.4f} recall={tp/(tp+fn):.4f} "
          f"approval={(tn+fn)/len(y_valid):.4f} "
          f"bad_approved={fp/(tn+fp):.4f}")

In [ ]:
try:
    import shap
    # Ambil model HGB dari pipeline
    hgb_clf = hgb.named_steps["clf"]
    X_valid_pre = hgb.named_steps["pre"].transform(X_valid)
    # Kolom hasil transform = num_cols + cat_cols
    feat_names = num_cols + cat_cols

    explainer = shap.Explainer(hgb_clf, X_valid_pre, feature_names=feat_names)
    shap_values = explainer(X_valid_pre, max_evals=500)

    shap.summary_plot(shap_values, X_valid_pre, feature_names=feat_names, max_display=25)
except ImportError:
    print("SHAP tidak tersedia, skip.")
except Exception as e:
    print("SHAP error:", e)
    print("Fallback: pakai feature importance dari HGB/RF")

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(hgb, X_valid, y_valid, n_repeats=5,
                               random_state=42, n_jobs=-1, scoring="roc_auc")
imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=False)
print(imp.head(25).to_string(index=False))

In [ ]:
!{sys.executable} -m pip install xgboost

In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
import xgboost as xgb
import numpy as np
import time
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

# --- Encode kategorikal jadi integer ---
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_x = X_train.copy()
X_valid_x = X_valid.copy()
X_test_x  = X_test.copy()

X_train_x[cat_cols] = enc.fit_transform(X_train_x[cat_cols].astype(str))
X_valid_x[cat_cols] = enc.transform(X_valid_x[cat_cols].astype(str))
X_test_x[cat_cols]  = enc.transform(X_test_x[cat_cols].astype(str))

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {spw:.2f}")

# --- XGBoost dengan setup yang benar ---
xgb_model = xgb.XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=4,              # turun dari 5, kurang overfit
    min_child_weight=20,      # naik dari 10
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=5.0,
    gamma=0.5,
    objective="binary:logistic",
    eval_metric="auc",        # STRING, bukan list
    scale_pos_weight=spw,
    tree_method="hist",
    enable_categorical=False, # MATIKAN native categorical
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50,
)

t0 = time.time()
xgb_model.fit(
    X_train_x, y_train,
    eval_set=[(X_valid_x, y_valid)],
    verbose=100,
)
print(f"XGB trained in {time.time()-t0:.1f}s | best_iter={xgb_model.best_iteration}")

p_xgb_va = xgb_model.predict_proba(X_valid_x)[:, 1]
p_xgb_te = xgb_model.predict_proba(X_test_x)[:, 1]
evaluate(y_valid, p_xgb_va, "XGB-fix valid")
evaluate(y_test,  p_xgb_te, "XGB-fix test")

In [ ]:
# Ensemble: LR + RF + XGB-fix (HGB dibuang)
# Bobot manual dulu
p_blend_va = 0.20*p_lr_va + 0.50*p_rf_va + 0.30*p_xgb_va
p_blend_te = 0.20*p_lr_te + 0.50*p_rf_te + 0.30*p_xgb_te
evaluate(y_valid, p_blend_va, "blend-new valid")
evaluate(y_test,  p_blend_te, "blend-new test")

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

def neg_ks(weights, preds, y):
    w = np.abs(weights) / np.abs(weights).sum()
    p = sum(wi * pi for wi, pi in zip(w, preds))
    return -(2 * roc_auc_score(y, p) - 1)

# Buang HGB, pakai LR + RF + XGB-fix
preds_va = [p_lr_va, p_rf_va, p_xgb_va]
preds_te = [p_lr_te, p_rf_te, p_xgb_te]

res = minimize(neg_ks, x0=[0.3, 0.4, 0.3], args=(preds_va, y_valid),
               method="Nelder-Mead", options={"xatol": 1e-4, "maxiter": 500})
w_opt = np.abs(res.x) / np.abs(res.x).sum()
print("Bobot optimal:", dict(zip(["LR","RF","XGB"], w_opt.round(3))))

p_blend_va = sum(wi * pi for wi, pi in zip(w_opt, preds_va))
p_blend_te = sum(wi * pi for wi, pi in zip(w_opt, preds_te))
evaluate(y_valid, p_blend_va, "blend-3 valid")
evaluate(y_test,  p_blend_te, "blend-3 test")

In [ ]:
from sklearn.isotonic import IsotonicRegression

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_blend_va, y_valid)
p_blend_va_cal = iso.predict(p_blend_va)
p_blend_te_cal = iso.predict(p_blend_te)

evaluate(y_valid, p_blend_va_cal, "blend-3-cal valid")
evaluate(y_test,  p_blend_te_cal, "blend-3-cal test")

t_opt = best_threshold(y_valid, p_blend_va_cal, cost_fn=10, cost_fp=1)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

t_opt = best_threshold(y_valid, p_blend_va_cal, cost_fn=10, cost_fp=1)

# KS di test
fpr, tpr, _ = roc_curve(y_test, p_blend_te_cal)
ks_test = float(np.max(tpr - fpr))
fpr_v, tpr_v, _ = roc_curve(y_valid, p_blend_va_cal)
ks_valid = float(np.max(tpr_v - fpr_v))

print(f"\n=== KS FINAL ===")
print(f"KS valid : {ks_valid:.4f}")
print(f"KS test  : {ks_test:.4f}")

# Confusion matrix di test
yp_te = (p_blend_te_cal >= t_opt).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, yp_te, labels=[0,1]).ravel()
print(f"\nTest @ {t_opt:.3f}: TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"Precision={tp/(tp+fp):.4f} | Recall={tp/(tp+fn):.4f}")
print(f"Approval rate={(tn+fn)/len(y_test):.4f} | Bad rate approved={fp/(tn+fp):.4f}")

In [ ]:
res_df = pd.DataFrame(RESULTS)
print(res_df.round(4).to_string(index=False))

# Fokus KS
print("\n=== KS ranking ===")
ks_rank = res_df[res_df["name"].str.contains("valid|test")].sort_values("ks", ascending=False)
print(ks_rank[["name", "ks", "auc", "gini"]].round(4).to_string(index=False))

In [ ]:
# Cek apakah XGB-fix sudah dijalankan
try:
    print("XGB best_iter:", xgb_model.best_iteration)
    print("eval_metric:", xgb_model.eval_metric)
    print("enable_categorical:", xgb_model.enable_categorical)
except NameError:
    print("XGB-fix BELUM dijalankan")

In [ ]:
p_xgb_va = xgb_model.predict_proba(X_valid_x)[:, 1]
p_xgb_te = xgb_model.predict_proba(X_test_x)[:, 1]
evaluate(y_valid, p_xgb_va, "XGB-fix valid")
evaluate(y_test,  p_xgb_te, "XGB-fix test")

In [ ]:
import pickle

# Simpan state
with open("state_after_fe.pkl", "wb") as f:
    pickle.dump({
        "master": master,
        "feature_cols": feature_cols,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "RESULTS": RESULTS,
        "lr": lr_fe, "rf": rf_fe, "xgb": xgb_fe,
    }, f)
print("Saved.")